## LOGISTIC REGRESSION

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Data
df = pd.read_csv("final_preprocessed_dataset.csv")

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# 3. Vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. Training (GridSearch)
log_reg = LogisticRegression(max_iter=1000)
param_grid = {'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear']}
grid = GridSearchCV(log_reg, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train_vec, y_train)
best_model = grid.best_estimator_

# 5. Feature Importance (Now that best_model exists!)
feature_names = vectorizer.get_feature_names_out()
coefs = best_model.coef_[0]
top_positive = sorted(zip(coefs, feature_names), reverse=True)[:20]
print("Top Propaganda Words:", top_positive)

# 6. Evaluation
test_pred = best_model.predict(X_test_vec)
print(classification_report(y_test, test_pred))

Top Propaganda Words: [(np.float64(4.730595855500658), 'congress'), (np.float64(4.524576111242118), 'bjp'), (np.float64(3.784458324360813), 'plague'), (np.float64(3.7483795308418983), 'gandhi'), (np.float64(3.616931219272256), 'election'), (np.float64(3.4869144090979223), 'government'), (np.float64(3.2732124841429466), 'leader'), (np.float64(3.1396236405807985), 'issue'), (np.float64(3.0853519082429264), 'internet'), (np.float64(3.0823852882359435), 'years'), (np.float64(3.062921493297265), 'tejashwi'), (np.float64(3.036646578017389), 'muslim'), (np.float64(3.0342238949016185), 'inflation'), (np.float64(2.8413579374967197), 'violence'), (np.float64(2.8312797729517434), 'farmers'), (np.float64(2.7942417114305234), 'entire'), (np.float64(2.78998896804229), 'trump'), (np.float64(2.7649106852522296), 'terrorism'), (np.float64(2.7304887891462717), 'kangana'), (np.float64(2.6968907988206583), 'brainberriesxd')]
              precision    recall  f1-score   support

         0.0       0.85   

## BERT


In [13]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# 1. Load data
df = pd.read_csv("final_preprocessed_dataset.csv")
df["label"] = df["label"].astype(int)

print(df["label"].value_counts())

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# 2. Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512
    )

train_enc = tokenize(train_df["text"].tolist())
val_enc   = tokenize(val_df["text"].tolist())

# 3. Dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_enc, train_df["label"].values)
val_dataset   = Dataset(val_enc, val_df["label"].values)

# 4. Model
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 5. Metrics function (NEW)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# 6. Training config
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    num_train_epochs=4,
    weight_decay = 0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True
)

# 7. Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics   # important
)

# 8. Train
trainer.train()

# 9. Final evaluation
results = trainer.evaluate()
print(results)

# 10. Test on custom samples
texts = [
    "This government is destroying the nation deliberately",
    "The weather today is pleasant and sunny"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model.to(device)

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = bert_model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)

print(preds)
preds = torch.argmax(outputs.logits, dim=1)

print("Predictions:", preds)
bert_model.save_pretrained("propaganda_bert")
tokenizer.save_pretrained("propaganda_bert")

label
1    1352
0    1309
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.458333,0.355676,0.857411,0.860294,0.857143,0.863469
2,0.265713,0.433547,0.859287,0.861368,0.862963,0.859779
3,0.334466,0.555753,0.859287,0.864376,0.847518,0.881919
4,0.216589,0.621992,0.857411,0.862319,0.846975,0.878229


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.3555165231227875, 'eval_accuracy': 0.8574108818011257, 'eval_f1': 0.8602941176470589, 'eval_precision': 0.8571428571428571, 'eval_recall': 0.8634686346863468, 'eval_runtime': 17.0212, 'eval_samples_per_second': 31.314, 'eval_steps_per_second': 3.936, 'epoch': 4.0}
tensor([0, 0], device='cuda:0')
Predictions: tensor([0, 0], device='cuda:0')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('propaganda_bert/tokenizer_config.json', 'propaganda_bert/tokenizer.json')

##RoBERTa

In [14]:
!pip install transformers[torch] datasets evaluate accelerate -q

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# loading dataset
df = pd.read_csv('final_preprocessed_dataset.csv')
df['label'] = df['label'].astype(int)

# Stratified split => consistent label distribution
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

# tokenizaion
robert_model = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_fn, batched=True)

# model + hyperparams
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./roberta_propaganda_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, predictions)}

# initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# trai model
print("Starting Training...")
trainer.train()

# Eval (train Vs test)
print("\n" + "="*20 + " TRAINING SET REPORT " + "="*20)
train_res = trainer.predict(tokenized_datasets["train"])
train_preds = np.argmax(train_res.predictions, axis=-1)
print(classification_report(tokenized_datasets["train"]["label"], train_preds, target_names=["Non-Propaganda", "Propaganda"]))

print("\n" + "="*20 + " TESTING SET REPORT " + "="*20)
test_res = trainer.predict(tokenized_datasets["test"])
test_preds = np.argmax(test_res.predictions, axis=-1)
print(classification_report(tokenized_datasets["test"]["label"], test_preds, target_names=["Non-Propaganda", "Propaganda"]))

# overfitting check
train_f1 = f1_score(tokenized_datasets["train"]["label"], train_preds)
test_f1 = f1_score(tokenized_datasets["test"]["label"], test_preds)
print(f"\nOverfitting Gap (Train - Test F1): {train_f1 - test_f1:.4f}")

Map:   0%|          | 0/2128 [00:00<?, ? examples/s]

Map:   0%|          | 0/266 [00:00<?, ? examples/s]

Map:   0%|          | 0/267 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


==================== TRAINING SET REPORT ====================


                precision    recall  f1-score   support

Non-Propaganda       0.85      0.90      0.87      1047
    Propaganda       0.90      0.84      0.87      1081

      accuracy                           0.87      2128
     macro avg       0.87      0.87      0.87      2128
  weighted avg       0.87      0.87      0.87      2128


==================== TESTING SET REPORT ====================


                precision    recall  f1-score   support

Non-Propaganda       0.84      0.82      0.83       131
    Propaganda       0.83      0.85      0.84       136

      accuracy                           0.84       267
     macro avg       0.84      0.84      0.84       267
  weighted avg       0.84      0.84      0.84       267


Overfitting Gap (Train - Test F1): 0.0301


## SAVING MODEL

In [15]:
import joblib

# Saving the individual models
joblib.dump(log_reg, 'logistic_regression_model.pkl')
joblib.dump(bert_model, 'bert_model.pkl')
joblib.dump(robert_model, 'roberta_model.pkl')

print("Models saved successfully!")

Models saved successfully!


In [21]:
import joblib

# 1. Save Logistic (Check if you named it 'log_reg' or 'best_model')
try:
    joblib.dump(log_reg, 'logistic_regression_model.pkl')
    print("Logistic model saved!")
except NameError:
    joblib.dump(best_model, 'logistic_regression_model.pkl')
    print("Best_model (Logistic) saved!")

# 2. Save BERT
# (Run this immediately after your BERT training cell)
model.save_pretrained("./bert_model_dir")
tokenizer.save_pretrained("./bert_model_dir")
print("BERT saved!")

# 3. Save RoBERTa
# (Run this immediately after your RoBERTa training cell)
model.save_pretrained("./roberta_model_dir")
tokenizer.save_pretrained("./roberta_model_dir")
print("RoBERTa saved!")

Logistic model saved!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT saved!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa saved!


## CREATING ENSEMBLE

In [22]:
from transformers import AutoModelForSequenceClassification

# Load into unique names so we can average them
logistic_model = joblib.load('logistic_regression_model.pkl')
bert_finetuned = AutoModelForSequenceClassification.from_pretrained("./bert_model_dir")
roberta_finetuned = AutoModelForSequenceClassification.from_pretrained("./roberta_model_dir")

print("All models loaded into memory simultaneously for ensemble!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

All models loaded into memory simultaneously for ensemble!


In [27]:
import numpy as np
import joblib
import torch
from scipy.special import softmax
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, Trainer, AutoTokenizer

# --- 1. SETUP & LOAD (Global Scope) ---
print("Step 1: Loading models and tokenizers...")
try:
    # Use best_model if it's in memory, otherwise load from disk
    if 'best_model' in globals():
        logistic_model = best_model
    else:
        logistic_model = joblib.load('logistic_regression_model.pkl')

    bert_model = AutoModelForSequenceClassification.from_pretrained("./bert_model_dir")
    roberta_model = AutoModelForSequenceClassification.from_pretrained("./roberta_model_dir")

    # We need the specific tokenizers for the alignment step
    bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
except Exception as e:
    print(f"Error loading models: {e}. Check if files exist in your Colab sidebar.")

# --- 2. DATA ALIGNMENT ---
print("Step 2: Aligning test data across all models...")
# Use test_df (the 10% slice) to ensure all models predict on the same rows
test_texts = test_df['text'].tolist()
y_test_actual = test_df['label'].values

# LR Vectorization
X_test_lr = vectorizer.transform(test_texts)

# --- 3. PREDICTION ENGINES ---
def run_ensemble():
    print("Step 3: Generating probabilities...")

    # Logistic Regression Probs
    lr_probs = logistic_model.predict_proba(X_test_lr)

    # Transformer Probs (Using raw torch for speed and to avoid Trainer NameErrors)
    def get_transformer_probs(model, tokenizer, texts):
        model.eval()
        inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
        return softmax(logits.numpy(), axis=1)

    print("  > Processing BERT...")
    bert_probs = get_transformer_probs(bert_model, bert_tokenizer, test_texts)

    print("  > Processing RoBERTa...")
    roberta_probs = get_transformer_probs(roberta_model, roberta_tokenizer, test_texts)

    # --- 4. SOFT VOTING ---
    # Averaging the probabilities
    ensemble_probs = (lr_probs + bert_probs + roberta_probs) / 3
    final_preds = np.argmax(ensemble_probs, axis=1)

    return final_preds

# --- 4. EXECUTION ---
y_pred_ensemble = run_ensemble()

print("\n" + "="*25)
print("  FINAL ENSEMBLE REPORT")
print("="*25)
print(classification_report(y_test_actual, y_pred_ensemble, target_names=["Non-Propaganda", "Propaganda"]))

Step 1: Loading models and tokenizers...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step 2: Aligning test data across all models...
Step 3: Generating probabilities...
  > Processing BERT...
  > Processing RoBERTa...

  FINAL ENSEMBLE REPORT
                precision    recall  f1-score   support

Non-Propaganda       0.85      0.81      0.83       131
    Propaganda       0.82      0.86      0.84       136

      accuracy                           0.84       267
     macro avg       0.84      0.83      0.83       267
  weighted avg       0.84      0.84      0.84       267



In [28]:
import os
import joblib

# 1. Create a parent directory for the ensemble
ensemble_dir = "final_ensemble_model"
os.makedirs(ensemble_dir, exist_ok=True)

print(f"Saving ensemble components to {ensemble_dir}...")

# 2. Save Logistic Regression & Vectorizer
# We save the vectorizer too because the model is useless without it!
joblib.dump(logistic_model, os.path.join(ensemble_dir, 'logistic_model.pkl'))
joblib.dump(vectorizer, os.path.join(ensemble_dir, 'tfidf_vectorizer.pkl'))

# 3. Save BERT Model and Tokenizer
bert_path = os.path.join(ensemble_dir, 'bert_component')
bert_model.save_pretrained(bert_path)
bert_tokenizer.save_pretrained(bert_path)

# 4. Save RoBERTa Model and Tokenizer
roberta_path = os.path.join(ensemble_dir, 'roberta_component')
roberta_model.save_pretrained(roberta_path)
roberta_tokenizer.save_pretrained(roberta_path)

print("✅ All components saved successfully!")

Saving ensemble components to final_ensemble_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ All components saved successfully!


In [35]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# --- 1. DATASET SAMPLES (30 cases: 15 Prop / 15 Non-Prop) ---
prop_samples = test_df[test_df['label'] == 1].sample(15, random_state=1)
non_prop_samples = test_df[test_df['label'] == 0].sample(15, random_state=1)
dataset_mix = pd.concat([prop_samples, non_prop_samples]).sample(frac=1, random_state=42)

dataset_texts = dataset_mix['text'].tolist()
dataset_labels = dataset_mix['label'].tolist()

# --- 2. INDIAN CONTEXT CUSTOM CASES (10 cases) ---
custom_cases = [
    {"text": "The opposition is a gang of anti-nationals working with foreign powers to destroy India.", "label": 1},
    {"text": "Every true Indian must share this message to expose the toolkit designed to malign our image.", "label": 1},
    {"text": "Our leader has achieved more in 5 years than others did in 70 years of total darkness.", "label": 1},
    {"text": "Shocking truth: How the previous government sold our national secrets to the enemy!", "label": 1},
    {"text": "Support the new bill or admit that you hate the progress and sacrifice of our brave soldiers.", "label": 1},
    {"text": "The Reserve Bank of India kept the repo rate unchanged at 6.5% during the latest review.", "label": 0},
    {"text": "Indian Railways announced a new Vande Bharat express route between Delhi and Jaipur.", "label": 0},
    {"text": "The Indian Meteorological Department predicts heavy rainfall in coastal Andhra Pradesh.", "label": 0},
    {"text": "A new startup incubator was inaugurated at Jaypee Institute to support young entrepreneurs.", "label": 0},
    {"text": "The prices of onions in the local mandi have stabilized after the arrival of fresh harvest.", "label": 0}
]

all_test_texts = dataset_texts + [c['text'] for c in custom_cases]
all_true_labels = dataset_labels + [c['label'] for c in custom_cases]

# --- 3. EXECUTION ---
# Re-running prediction to ensure all context is fresh
predictions = predict_ensemble(all_test_texts, components)

# --- 4. FORMATTED DASHBOARD (No Accuracy, Custom Labels) ---
results_df = pd.DataFrame({
    'Text': all_test_texts,
    'True Label': all_true_labels,
    'Predicted': predictions,
    'Correct': np.array(all_true_labels) == np.array(predictions)
})

def display_clean_dashboard(df):
    html = f"<h2 style='color: #1a202c; font-family: Arial;'>Ensemble Model Testing: Propaganda vs Non-Propaganda</h2>"
    html += "<table style='width:100%; border: 1px solid #cbd5e0; border-collapse: collapse; font-family: Arial; font-size: 13px;'>"
    html += "<tr style='background-color: #2d3748; color: white; text-align: left;'>"
    html += "<th style='padding: 10px; border: 1px solid #cbd5e0;'>Result</th>"
    html += "<th style='padding: 10px; border: 1px solid #cbd5e0;'>Test Case Text</th>"
    html += "<th style='padding: 10px; border: 1px solid #cbd5e0;'>Actual Label</th>"
    html += "<th style='padding: 10px; border: 1px solid #cbd5e0;'>Model Prediction</th></tr>"

    for _, row in df.iterrows():
        bg_color = "#f0fff4" if row['Correct'] else "#fff5f5"
        status = "MATCH" if row['Correct'] else "MISMATCH"

        # Label remapping
        actual = "PROPAGANDA" if row['True Label'] == 1 else "NON-PROPAGANDA"
        pred = "PROPAGANDA" if row['Predicted'] == 1 else "NON-PROPAGANDA"

        # Color logic
        a_color = "#c53030" if row['True Label'] == 1 else "#2b6cb0"
        p_color = "#c53030" if row['Predicted'] == 1 else "#2b6cb0"

        html += f"<tr style='background-color: {bg_color};'>"
        html += f"<td style='padding: 10px; border: 1px solid #cbd5e0; font-weight: bold;'>{status}</td>"
        html += f"<td style='padding: 10px; border: 1px solid #cbd5e0;'>{row['Text'][:110]}...</td>"
        html += f"<td style='padding: 10px; border: 1px solid #cbd5e0; color: {a_color}; font-weight: bold;'>{actual}</td>"
        html += f"<td style='padding: 10px; border: 1px solid #cbd5e0; color: {p_color}; font-weight: bold;'>{pred}</td>"
        html += "</tr>"

    html += "</table>"
    display(HTML(html))

display_clean_dashboard(results_df)

  > Extracting BERT features...
  > Extracting RoBERTa features...


Result,Test Case Text,Actual Label,Model Prediction
MATCH,new delhi december after many years of demand and discussion the post of chief of defense staff was created in...,NON-PROPAGANDA,NON-PROPAGANDA
MATCH,omicron in india the omicron variant of corona virus is spreading rapidly in the country omicron has also made...,NON-PROPAGANDA,NON-PROPAGANDA
MATCH,the girl had asked for help after gangrape but the ngo hid the informationxd by deepak kumar pant published de...,NON-PROPAGANDA,NON-PROPAGANDA
MATCH,national desk chief of defense staff general bipin rawat his wife madhulika rawat and others have died in a he...,NON-PROPAGANDA,NON-PROPAGANDA
MATCH,uproar over mamita mehar murder case continues for fourth day odisha assembly proceedings adjournedxd ever sin...,PROPAGANDA,PROPAGANDA
MATCH,chandigarh former punjab chief minister captain amarinder singh said on monday that his partys aim is to win t...,PROPAGANDA,PROPAGANDA
MATCH,he further said that it is too early to say how contagious it can be the doctor said we dont know how serious ...,NON-PROPAGANDA,NON-PROPAGANDA
MATCH,pakistan is not desisting from its nefarious activities he keeps planning to harm india in some way or the oth...,NON-PROPAGANDA,NON-PROPAGANDA
MISMATCH,indian tennis star leander paes joined tmc on friday paes who has gained international recognition as a tennis...,PROPAGANDA,NON-PROPAGANDA
MATCH,patna the last day of the winter session of bihar legislative assembly ended with controversy during the concl...,PROPAGANDA,PROPAGANDA
